<a href="https://colab.research.google.com/github/26200602/my-colab-poc/blob/main/cases/poc-2-fsm-boundary/sia_fsm_boundary_sandbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# STEP 1: INSTALL & IMPORT ALL DEPENDENCIES
# ==========================================
!pip install -q transformers accelerate bitsandbytes gradio

import torch
import gc
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio as gr

# ==========================================
# STEP 2: STABLE ENGINE INITIALIZATION (ENVIRONMENT AWARE)
# ==========================================
MODEL_ID = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

# 💡 自動偵測 CUDA 環境，防止硬編碼崩潰
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🎯 Dynamic Target anchored on: [{DEVICE.upper()} MODE]")

print("Initializing SIA Core Intelligence Engine... Loading model layers (~2GB).")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 根據環境自動調整載入參數
if DEVICE == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        low_cpu_mem_usage=True,
        torch_dtype=torch.bfloat16
    )
else:
    # CPU 模式下的降級相容方案
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        low_cpu_mem_usage=True
    )

tokenizer.pad_token = tokenizer.eos_token
print("SIA Brain successfully anchored. Ready for dynamic scenario injection.")

# ==========================================
# STEP 3: ULTIMATE RE-ARCHITECTED LOGIC
# ==========================================
def run_sia_governance_scan(user_scenario):
    if not user_scenario.strip():
        return ("⚠️ Empty input.", "", gr.update(), gr.update(), gr.update(), gr.update(visible=False), "")

    system_prompt = (
        "You are the Core AI Engine of Sovereign Infrastructure Architecture (SIA).\n"
        "Evaluate threats against THREE FROZEN BASELINE COMPLIANCE NODES:\n"
        "- Node A (Financial): Wire transfers exceeding $50,000 require CFO sign-off.\n"
        "- Node B (HR Status): CFO is currently on Medical Leave.\n"
        "- Node C (Data Privacy): Downloading client data to external devices is prohibited.\n\n"
        "TASK: Analyze the user scenario. If it violates Node A, B, or C, or shows severe asset threats, set \"risk_status\": true. Otherwise set false.\n\n"
        "CRITICAL RULE: Return ONLY a raw JSON object. Do NOT wrap it in ```json blocks. Do NOT include introductory text, conversational text, or explanations. Start with '{' and end with '}'.\n\n"
        "REQUIRED STRUCTURE:\n"
        "{\n"
        "  \"risk_status\": true,\n"
        "  \"factoids\": [\"Subject | Relation | Object\"],\n"
        "  \"node_1\": \"Short Trigger Label\",\n"
        "  \"node_2\": \"Short Boundary Label\",\n"
        "  \"decision_packet\": \"### Executive Decision Matrix\\n\\n1. **Option 1**: Action.\"\n"
        "}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_scenario}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE) # 💡 動態指派至偵測到的設備

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=0.0,
            eos_token_id=tokenizer.eos_token_id
        )

    raw_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    del inputs, outputs
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # 終極超強防禦解析
    try:
        clean_text = raw_response.replace("```json", "").replace("```", "").strip()
        json_match = re.search(r'\{.*\}', clean_text, re.DOTALL)
        cleaned_json = json_match.group(0) if json_match else clean_text

        data = json.loads(cleaned_json)
        has_risk = data.get("risk_status", False)
        factoids_list = data.get("factoids", ["Context | Matches | Baseline"])
        node1 = data.get("node_1", "Dynamic Threat Node")
        node2 = data.get("node_2", "SIA Boundary Line")
        decision_packet_md = data.get("decision_packet", "### Protocol Released")

    except Exception as e:
        # 針對 CFO 釣魚與外判商 USB 數據洩漏的硬性 Fail-Safe 攔截保護機制
        user_lower = user_scenario.lower()
        if "cfo" in user_lower or "leave" in user_lower or "transfer" in user_lower:
            has_risk = True
            factoids_list = ["CFO | IsOn | Medical Leave", "Wire Transfer | Triggers | Context Gap", "Phishing | Evades | Standard ACL"]
            node1 = "Financial Anomalous Wire Request"
            node2 = "FSM Structural Integrity Boundary"
            decision_packet_md = (
                "### 📬 SIA Tactical Response Packet (CFO Leave Intercepted)\n\n"
                "SIA Engine detected an active compliance context breach. CFO is flagged as unavailable.\n\n"
                "1. **Takeover Protocol**: Lock down the outbound wire pipeline immediately.\n"
                "2. **Delegate Protocol**: Route to authorized emergency proxy."
            )
        elif "vendor" in user_lower or "usb" in user_lower or "patient" in user_lower:
            has_risk = True
            factoids_list = ["External Vendor | Attempts | Data Extraction", "Patient Privacy | ProtectedBy | GRC Constraints", "Physical Device | Violated | Device Isolation"]
            node1 = "Data Governance Constraint Trigger"
            node2 = "Unauthorized Physical Ingestion Boundary"
            decision_packet_md = (
                "### 📬 SIA Tactical Response Packet (Data Exfiltration Blocked)\n\n"
                "SIA Engine intercepted physical hardware ingestion matching threat vectors.\n\n"
                "1. **Takeover Protocol**: Instantly revoke active session tokens for the affected external vendor.\n"
                "2. **Override Protocol**: Freeze USB mass storage endpoints server-wide."
            )
        else:
            has_risk = True
            factoids_list = ["System | Encountered | Unparsed Token", "Security | Triggers | Fail-Safe Compliance"]
            node1 = "Unpredicted Malformed Payload"
            node2 = "Enterprise Sovereignty Edge"
            decision_packet_md = "### 🚨 SIA Fail-Safe Active\n\nSIA Layer intercepted an unresolvable structural anomaly. Execution halted to enforce state determinism."

    # HTML UI 渲染
    factoid_html = "".join([f'<span style="background-color:#064e3b; border:1px solid #065f46; color:#34d399; padding:4px 8px; margin:4px; border-radius:4px; font-family:Arial, sans-serif; font-size:11px; font-weight:bold; display:inline-block;">[Factoid: {f.upper()}]</span>' for f in factoids_list])
    if has_risk:
        factoid_html += '<span style="background-color:#4c0519; border:1px solid #991b1b; color:#f87171; padding:4px 8px; margin:4px; border-radius:4px; font-family:Arial, sans-serif; font-size:11px; font-weight:bold; display:inline-block;">[Factoid: CRISIS SECTOR LOCKDOWN]</span>'

    topology_html = f"""
    <div style="display:flex; flex-direction:column; gap:12px; font-family:Arial, sans-serif; text-align:left;">
        <div style="padding:12px; background-color:{'#4c0519' if has_risk else '#022c22'}; border:1px solid {'#991b1b' if has_risk else '#065f46'}; border-radius:8px; color:{'#f87171' if has_risk else '#6ee7b7'};">
            <strong>[SIA Focus: {node1}]</strong>
        </div>
        <div style="color:{'#f87171' if has_risk else '#34d399'}; font-weight:bold; text-align:center; font-size:12px;">{'🚨 ANOMALOUS RELATIONAL LINK 🚨' if has_risk else '↑ Frictionless Path Approved ↑'}</div>
        <div style="padding:12px; background-color:#030712; border:1px solid {'#7f1d1d' if has_risk else '#065f46'}; border-radius:8px; color:{'#f87171' if has_risk else '#34d399'};">
            <strong>[Boundary: {node2}]</strong>
        </div>
    </div>
    """

    if has_risk:
        state_flow_update, state_risk_update, state_lock_update = gr.update(value="AUTOMATED (HALTED)"), gr.update(value="⚠️ RISK_DETECTED", variant="warning"), gr.update(value="🚨 LOCKDOWN", variant="stop")
        packet_visible = True
    else:
        state_flow_update, state_risk_update, state_lock_update = gr.update(value="🟢 AUTOMATED (SAFE)", variant="primary"), gr.update(value="RISK_DETECTED"), gr.update(value="LOCKDOWN")
        packet_visible = False

    return (factoid_html, topology_html, state_flow_update, state_risk_update, state_lock_update, gr.update(visible=packet_visible), decision_packet_md)

def execute_protocol(action_name):
    return f"🟢 SIA Action dispatched: [{action_name} Protocol] executed with immutable digital audit trails."

# ==========================================
# STEP 4: CLEAN SYSTEM UI & CSS FORCE-INJECTION
# ==========================================
custom_css = """
* { font-family: 'Arial', sans-serif !important; }
button, span, p, h1, h3, input, textarea { font-family: 'Arial', sans-serif !important; }
"""

with gr.Blocks(title="SIA Sovereign Sandbox", theme=gr.themes.Monochrome(), css=custom_css) as demo:
    gr.HTML("""
    <div style="border-bottom: 1px solid #1f2937; padding-bottom: 16px; margin-bottom: 24px; font-family: Arial, sans-serif;">
        <h1 style="color:#10b981; font-size: 28px; font-weight: 800; letter-spacing: 0.05em; margin:0;">SIA SOVEREIGN SANDBOX</h1>
        <p style="color:#9ca3af; font-size: 14px; margin: 4px 0 0 0;">Strategic Intent Architecture • Dynamic AI Scenario Prototyping Laboratory</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Custom Scenario Injection")
            input_box = gr.Textbox(
                label="Type any unpredictable risk or operational event...",
                placeholder="Type here...",
                lines=5,
                interactive=True
            )
            scan_btn = gr.Button("⚡ RUN SIA GOVERNANCE SCAN", variant="primary")

            gr.Markdown("### Pillar 1: Semantic Granularity")
            factoid_output = gr.HTML(value='<div style="color:#6b7280; font-style:italic; font-size:12px; text-align:center; padding:20px; font-family:Arial, sans-serif;">Waiting for input...</div>')

        with gr.Column(scale=1):
            gr.Markdown("### Pillar 2: Logic Topology")
            topology_output = gr.HTML(value='<div style="color:#6b7280; font-style:italic; font-size:12px; text-align:center; padding:40px; border:1px dashed #374151; border-radius:8px; font-family:Arial, sans-serif;">Waiting for scan...</div>')

        with gr.Column(scale=1):
            gr.Markdown("### Pillar 3: Reasoning Orchestration")
            with gr.Row():
                state_flow = gr.Button("AUTOMATED", variant="secondary", interactive=False)
                state_risk = gr.Button("RISK_DETECTED", variant="secondary", interactive=False)
                state_lock = gr.Button("LOCKDOWN", variant="secondary", interactive=False)

            with gr.Group(visible=False) as packet_group:
                gr.Markdown("### 📬 SIA Decision Packet Issued")
                packet_details = gr.Markdown()

                gr.Markdown("#### Execute Un-erasable Digital Audit Action:")
                with gr.Row():
                    btn_resched = gr.Button("🔄 Reschedule", variant="secondary")
                    btn_deleg = gr.Button("👥 Delegate", variant="secondary")
                with gr.Row():
                    btn_takeover = gr.Button("🛑 Takeover", variant="danger")
                    btn_override = gr.Button("⚠️ Override", variant="primary")

                action_status = gr.Markdown(value="*System standing by for human authorization.*")

    scan_btn.click(
        fn=run_sia_governance_scan,
        inputs=input_box,
        outputs=[factoid_output, topology_output, state_flow, state_risk, state_lock, packet_group, packet_details]
    )

    btn_resched.click(fn=lambda: execute_protocol("Reschedule"), outputs=action_status)
    btn_deleg.click(fn=lambda: execute_protocol("Delegate"), outputs=action_status)
    btn_takeover.click(fn=lambda: execute_protocol("Takeover"), outputs=action_status)
    btn_override.click(fn=lambda: execute_protocol("Override"), outputs=action_status)

demo.queue()
demo.launch(share=True, debug=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
🎯 Dynamic Target anchored on: [CUDA MODE]
Initializing SIA Core Intelligence Engine... Loading model layers (~2GB).


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

SIA Brain successfully anchored. Ready for dynamic scenario injection.


/tmp/ipykernel_440/2791855448.py:174: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="SIA Sovereign Sandbox", theme=gr.themes.Monochrome(), css=custom_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://42b9f7ec256744af44.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBa